This notebook takes data from the WRMD database and transforms it to conform to the specification
required to map in ArcGIS and join with the Animal Vehicle Collision Data from the WCV WildOne database

Our target schema follows, it's based on the WCV WildOne Data:

- [x] OrganizationName
- [x] CaseNumber
    - Will need to be calculated - {Year}-{admission ID}
- [x] PatientID
- [x] CommonSpeciesName
- [x] GeneralSpeciesName 
- [x] DVertebrate
    - (Bird, Mammal, Reptile, Amphibian)
- [x] DateAdmitted
- [x] DDateAdmittedYear
- [x] Season
- [x] DDateAdmittedMonth
- [x] DDateAdmittedDOM
- [x] DDateAdmittedDOW
- [removed] CircumstancesOfRescue - Remove
- [removed] RescueState - Remove
- [removed] RescueJurisdiction - Remove
- [removed] RescueAddress - Remove
- [removed] OtherRescueInformation - Remove
- [x] Latitude
- [x] Longitude
- [removed] Elevation - Remove
- [x] Disposition
    - This is the Original 
- [x] UpdatedDisposition
    - Unified language (Active, Died, Released, Transferred). Rename this to just Disposition and drop the other one
- [x] DayOfWeekNumber 
- [x] MonthNumber

TODO:
- [x] Check with Linda about the organization name - is there a way to query it from WRMD
- [x] See about how to get unique identifiers for each record in WRMD
- [x] Double check the common names against the existing wcv common names for consistency
- [x] Use the CollisionAnimalMapping CSV to create the General Species Name column, check if all common species exist in it or we need to add more mappings
- [x] CollisionAnimalMapping CSV is updated after merge from main, revise the column

In [1]:
import pandas as pd
# Load in pickled dataframe
df = pd.read_pickle("./datasets/WRMD_2016_to_2024_inside_va_geocoding_not_required.pkl")

In [2]:
df.head() 

,admissions.case_year,admissions.hash,admissions.id,exams.age,exams.age_unit,exams.attitude,exams.bcs,exams.body,exams.cardiopulmonary,exams.cns,...,index_right,STATEFP,STATENS,AFFGEOID,GEOID,STUSPS,NAME,LSAD,ALAND,AWATER
0,2016,NaN,1,0.0,Adult,Depressed,Good,NaN,congested breathing,"unable to stay sternal, ataxia",...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
2,2016,NaN,3,0.0,Adult,Alert,Emaciated,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
8,2016,NaN,9,0.0,Adult,Depressed,Thin,NaN,NaN,barely responsive,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
10,2016,NaN,11,0.0,Adult,Quiet,Thin,NaN,NaN,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09
16,2016,NaN,17,0.0,Adult,Quiet,Reasonable,NaN,rapid and shallow breathing,NaN,...,18.0,51,01779803,0400000US51,51,VA,Virginia,00,1.022577e+11,8.528532e+09


In [3]:
# List all the columns in the dataframe
print("Columns in the DataFrame:")
for column in df.columns:
    print(column)

Columns in the DataFrame:
admissions.case_year
admissions.hash
admissions.id
exams.age
exams.age_unit
exams.attitude
exams.bcs
exams.body
exams.cardiopulmonary
exams.cns
exams.comments
exams.dehydration
exams.examined_at
exams.examiner
exams.forelimb
exams.gastrointestinal
exams.head
exams.hindlimb
exams.integument
exams.mm_color
exams.mm_texture
exams.musculoskeletal
exams.nutrition
exams.sex
exams.temperature
exams.temperature_unit
exams.treatment
exams.type
exams.weight
exams.weight_unit
patient_locations.area
patient_locations.comments
patient_locations.enclosure
patient_locations.moved_in_at
patient_locations.where_holding
patients.address_found
patients.admitted_at
patients.admitted_by
patients.band
patients.carcass_saved
patients.care_by_rescuer
patients.city_found
patients.clinical_signs
patients.common_name
patients.county_found
patients.criminal_activity
patients.custom_field_1
patients.custom_field_2
patients.days_in_care
patients.diagnosis
patients.disposition
patients.disp

## Column by Column Transformations

### Organization Name

In [4]:
# Add a column to the dataframe named "OrganizationName" and fill it with the value "WRMD"
df["OrganizationName"] = "WRMD"


### Case Number

In [5]:
# Create a new columns called "Case Number" where the value is based on the "case year" and "id" columns
df["Case Number"] = df["OrganizationName"] + df["admissions.case_year"].astype(str) + "-" + df["admissions.id"].astype(str)

In [6]:
df["Case Number"]


0           WRMD2016-1
2           WRMD2016-3
8           WRMD2016-9
10         WRMD2016-11
16         WRMD2016-17
             ...      
24955    WRMD2024-3752
24996    WRMD2024-3793
25024    WRMD2024-3821
25049    WRMD2024-3846
25058    WRMD2024-3855
Name: Case Number, Length: 1093, dtype: object

### PatientID

In [7]:
# Create a column called PatientID where the value is empty
df["PatientID"] = None

### CommonSpeciesName

In [8]:
# Create a columnn called CommonSpeciesName based on the patient.common_name column
df["CommonSpeciesName"] = df["patients.common_name"].str.lower()

### GeneralSpeciesName

In [9]:
# We need to check that every unique value in the CommonSpeciesName column is in the CollisionAnimalMapping.csv file

collision_mapping = pd.read_csv("./lookup_tables/CollisionAnimalMapping.csv")
# Check if all unique values in CommonSpeciesName are in the collision_mapping
unique_common_names = df["CommonSpeciesName"].unique()
missing_names = set(unique_common_names) - set(collision_mapping["Animal"].str.lower())
assert len(missing_names) == 0, f"Missing common names in collision mapping: {missing_names}"

In [10]:
# Create a column called GeneralSpeciesName based on looking up the CommonSpeciesName in the CollisionAnimalMapping.csv file
collision_mapping["Animal"] = collision_mapping["Animal"].str.lower()
df["GeneralSpeciesName"] = df["CommonSpeciesName"].map(
    dict(zip(collision_mapping["Animal"], collision_mapping["Mapping"]))
)

In [11]:
df[['GeneralSpeciesName', 'CommonSpeciesName']]
# assert that if CommonSpeciesName is not null, GeneralSpeciesName is not null
assert df[df["CommonSpeciesName"].notnull()]["GeneralSpeciesName"].notnull().all(), "Some CommonSpeciesName values are missing a GeneralSpeciesName"

### DVertebrate

In [12]:
# We need to check that every unique value in the CommonSpeciesName column is in the AnimalsCategorized.csv file
animals_categorized = pd.read_csv("./lookup_tables/AnimalsCategorized.csv")
# Check if all unique values in CommonSpeciesName are in the animals_categorized
unique_common_names = df["CommonSpeciesName"].unique()
missing_names = set(unique_common_names) - set(animals_categorized["CommonSpeciesName"].str.lower())
assert len(missing_names) == 0, f"Missing common names in animals categorized: {missing_names}"

In [13]:
# Create a column called Dvertebrate based on looking up CommonSpeciesName in the AnimalsCategorized.csv file
animals_categorized["CommonSpeciesName"] = animals_categorized["CommonSpeciesName"].str.lower()
df["DVertebrate"] = df["CommonSpeciesName"].map(
    dict(zip(animals_categorized["CommonSpeciesName"], animals_categorized["DVertebrate"]))
)

In [14]:
# assert that if CommonSpeciesName is not null, GeneralSpeciesName is not null
assert df[df["CommonSpeciesName"].notnull()]["DVertebrate"].notnull().all(), "Some CommonSpeciesName values are missing a DVertebrate"
df[['DVertebrate', 'CommonSpeciesName']]

,DVertebrate,CommonSpeciesName
0,Mammal,virginia opossum
2,Bird,black vulture
8,Bird,barred owl
10,Bird,barred owl
16,Bird,american robin
...,...,...
24955,Bird,red-tailed hawk
24996,Bird,red-tailed hawk
25024,Bird,red-tailed hawk
25049,Bird,red-shouldered hawk


### DateAdmitted

In [15]:
# Check that the "patients.admitted_at" column contains no null values
assert df["patients.admitted_at"].notnull().all(), "There are null values in the patients.admitted_at column"

In [16]:
df["patients.admitted_at"]

0        2016-01-01 20:36:00
2        2016-01-06 23:30:00
8        2016-01-14 16:00:00
10       2016-01-15 17:50:00
16       2016-01-19 19:00:00
                ...         
24955    2024-11-21 14:49:00
24996    2024-12-01 15:48:00
25024    2024-12-08 15:44:00
25049    2024-12-16 10:18:00
25058    2024-12-20 10:05:00
Name: patients.admitted_at, Length: 1093, dtype: object

In [17]:
# Fill the "DateAdmitted" column with the values from "patients.admitted_at" column in the format MM/DD/YYYY
df["patients.admitted_at"] = pd.to_datetime(df["patients.admitted_at"], errors='coerce')
df["DateAdmitted"] = df["patients.admitted_at"].dt.strftime("%m/%d/%Y")

In [18]:
assert df["DateAdmitted"].notnull().all(), "There are null values in the patients.admitted_at column"

In [19]:
df[["DateAdmitted", "patients.admitted_at"]]

,DateAdmitted,patients.admitted_at
0,01/01/2016,2016-01-01 20:36:00
2,01/06/2016,2016-01-06 23:30:00
8,01/14/2016,2016-01-14 16:00:00
10,01/15/2016,2016-01-15 17:50:00
16,01/19/2016,2016-01-19 19:00:00
...,...,...
24955,11/21/2024,2024-11-21 14:49:00
24996,12/01/2024,2024-12-01 15:48:00
25024,12/08/2024,2024-12-08 15:44:00
25049,12/16/2024,2024-12-16 10:18:00


### DDateAdmittedYear

In [20]:
# Fill the "DDateAdmittedYear" column with the year from "patients.admitted_at" column
df["DDateAdmittedYear"] = df["patients.admitted_at"].dt.year

In [21]:
assert df["DDateAdmittedYear"].notnull().all(), "There are null values in the DDateAdmittedYear column"

### Season

In [22]:
'''
Winter: December 22 - March 20
Spring: March 21 - June 20
Summer: June 21 - September 22
Autumn: September 23 - December 21
'''
# Using the 'patients.admitted_at' column, create a new column called 'SeasonAdmitted' based on the date ranges above
def get_season(date):
    if date.month == 12 and date.day >= 22 or date.month in [1, 2] or (date.month == 3 and date.day <= 20):
        return "Winter"
    elif (date.month == 3 and date.day >= 21) or date.month in [4, 5] or (date.month == 6 and date.day <= 20):
        return "Spring"
    elif (date.month == 6 and date.day >= 21) or date.month in [7, 8] or (date.month == 9 and date.day <= 22):
        return "Summer"
    else:
        return "Autumn"
df["Season"] = df["patients.admitted_at"].apply(get_season)

In [23]:
assert df["Season"].notnull().all(), "There are null values in the DDateAdmittedYear column"
# Assert that the Season column only contains the values Winter, Spring, Summer, Autumn
valid_seasons = {"Winter", "Spring", "Summer", "Autumn"}
assert set(df["Season"].unique()).issubset(valid_seasons), f"Invalid seasons found: {set(df['Season'].unique()) - valid_seasons}"

### DDateAdmittedMonth

In [24]:
# Determine the month from the 'patients.admitted_at' column and create a new column called 'DDateAdmittedMonth' that has the months name
df["DDateAdmittedMonth"] = df["patients.admitted_at"].dt.month_name()

In [25]:
df["DDateAdmittedMonth"]

0         January
2         January
8         January
10        January
16        January
           ...   
24955    November
24996    December
25024    December
25049    December
25058    December
Name: DDateAdmittedMonth, Length: 1093, dtype: object

### DDateAdmittedDOM

In [26]:
# Create a column, DDateAdmittedDOM, based on the number of the day of the month from the 'patients.admitted_at' column
df["DDateAdmittedDOM"] = df["patients.admitted_at"].dt.day

In [27]:
df["DDateAdmittedDOM"]

0         1
2         6
8        14
10       15
16       19
         ..
24955    21
24996     1
25024     8
25049    16
25058    20
Name: DDateAdmittedDOM, Length: 1093, dtype: int32

### DDateAdmittedDOW

In [28]:
# Create a column called DDateAdmittedDOW and fill it with the name of the day of the week from the 'patients.admitted_at' column
# Fix this to use the three letter abbreviation for the day of the week
df["DDateAdmittedDOW"] = df["patients.admitted_at"].dt.day_name()

### Latitude

In [ ]:
df["Latitude"] = df["patients.lat_found"]

In [ ]:
df['Latitude']

0        38.960822
2        39.325379
8        39.219637
10       39.400742
16       39.041987
           ...    
24955    39.256726
24996    38.912531
25024    39.080584
25049    38.377775
25058    38.847600
Name: y, Length: 1093, dtype: float64

### Longitude

In [ ]:
df["Longitude"] = df["patients.lng_found"]

In [ ]:
df['Longitude']

0       -77.741824
2       -77.738882
8       -77.523534
10      -78.115627
16      -77.605404
           ...    
24955   -78.101397
24996   -77.921283
25024   -78.216929
25049   -77.459324
25058   -77.996685
Name: x, Length: 1093, dtype: float64

### Disposition

In [48]:
df["Disposition"] = df["patients.disposition"].str.lower()

### UpdatedDisposition

In [49]:
df["UpdatedDisposition"] = df["patients.disposition"].str.lower()

In [50]:
# List all unique values in the Disposition column
unique_dispositions = df["UpdatedDisposition"].unique()
print("Unique Dispositions:")
for disposition in unique_dispositions:
    print(disposition)

Unique Dispositions:
died in 24hr
euthanized in 24hr
released
euthanized +24hr
transferred
died +24hr
dead on arrival
pending


In [51]:
"""
Active, Died, Released, Transferred
"""
# if the disposition contains the string "died" or "euthanized", set the disposition to "Died"
df.loc[df["Disposition"].str.contains("died|euthanized|dead", case=False, na=False), "Disposition"] = "Died"
# if the disposition contains the string "released", set the disposition to "Released"
df.loc[df["Disposition"].str.contains("released", case=False, na=False), "Disposition"] = "Released"
# if the disposition contains the string "transferred", set the disposition to "Transferred"
df.loc[df["Disposition"].str.contains("transferred", case=False, na=False), "Disposition"] = "Transferred"
# if the disposition contains the string "pending", set the disposition to "Active"
df.loc[df["Disposition"].str.contains("pending", case=False, na=False), "Disposition"] = "Active"

In [52]:
# assert that the UpdatedDisposition column only contains the values Active, Died, Released, Transferred
valid_dispositions = {"Active", "Died", "Released", "Transferred"}
assert set(df["Disposition"].unique()).issubset(valid_dispositions), f"Invalid dispositions found: {set(df['Disposition'].unique()) - valid_dispositions}"

### DayOfWeekNumber 

In [53]:
# create a column called DayOfWeekNumber based on the day of the week number from the 'patients.admitted_at' column
df["DayOfWeekNumber"] = df["patients.admitted_at"].dt.dayofweek

In [54]:
df["DayOfWeekNumber"]

0        4
2        2
8        3
10       4
16       1
        ..
24955    3
24996    6
25024    6
25049    0
25058    4
Name: DayOfWeekNumber, Length: 1093, dtype: int32

### MonthNumber

In [55]:
# Create a column called MonthNumber based on the month number from the 'patients.admitted_at' column
df["MonthNumber"] = df["patients.admitted_at"].dt.month

In [56]:
df["MonthNumber"]

0         1
2         1
8         1
10        1
16        1
         ..
24955    11
24996    12
25024    12
25049    12
25058    12
Name: MonthNumber, Length: 1093, dtype: int32

## Outputing Transformed Data

In [ ]:
cols_to_keep = [
    "OrganizationName",
    "Case Number",
    "PatientID",
    "CommonSpeciesName",
    "GeneralSpeciesName",
    "DVertebrate",
    "DateAdmitted",
    "DDateAdmittedYear",
    "Season",
    "DDateAdmittedMonth",
    "DDateAdmittedDOM",
    "DDateAdmittedDOW",
    "Latitude",
    "Longitude",
    "Disposition",
    "UpdatedDisposition",
    "DayOfWeekNumber",
    "MonthNumber",
]

In [58]:
# Output the dataframe into a csv file
output_file = "./datasets/WRMD_2016_to_2024_geocoding_not_req_transformed.csv"
df[cols_to_keep].to_csv(output_file, index=False)